In [1]:
from __future__ import annotations

import numpy as np
import openml
import plotly.express as px
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

In [2]:
SEED=0

In [4]:
# Load the TabArena-v0.1 suite
suite = openml.study.get_suite(457)
assert suite.tasks is not None

for task_id in suite.tasks:
    task = openml.tasks.get_task(task_id)
    dataset = task.get_dataset()
    x, y, categorical_mask, feature_names = dataset.get_data(
        dataset_format="dataframe",
        target=dataset.default_target_attribute
    )
    print(f"{dataset.name}: {x.shape}")

airfoil_self_noise: (1503, 5)
Amazon_employee_access: (32769, 9)
anneal: (898, 38)
Another-Dataset-on-used-Fiat-500: (1538, 7)
APSFailure: (76000, 170)
bank-marketing: (45211, 13)
Bank_Customer_Churn: (10000, 10)
Bioresponse: (3751, 1776)
blood-transfusion-service-center: (748, 4)
churn: (5000, 19)
coil2000_insurance_policies: (9822, 85)
concrete_compressive_strength: (1030, 8)
credit-g: (1000, 20)
credit_card_clients_default: (30000, 23)
customer_satisfaction_in_airline: (129880, 21)
diabetes: (768, 8)
Diabetes130US: (71518, 47)
diamonds: (53940, 9)
E-CommereShippingData: (10999, 10)
Fitness_Club: (1500, 6)
Food_Delivery_Time: (45451, 9)
GiveMeSomeCredit: (150000, 10)
hazelnut-spread-contaminant-detection: (2400, 30)
healthcare_insurance_expenses: (1338, 6)
heloc: (10459, 23)
hiva_agnostic: (3845, 1617)
houses: (20640, 8)
HR_Analytics_Job_Change_of_Data_Scientists: (19158, 12)
in_vehicle_coupon_recommendation: (12684, 24)
Is-this-a-good-customer: (1723, 13)
kddcup09_appetency: (50000,

In [ ]:
def split_train_val_test(x, y, train_ratio: float, test_ratio: float):
    total = train_ratio + test_ratio

    if total < 0.5 or total > 0.95:
        raise ValueError("train_ratio + test_ratio must be between 0.5 and 0.95")

    x_train, x_temp, y_train, y_temp = train_test_split(
        x,
        y,
        test_size=1 - train_ratio,
        random_state=SEED
    )

    # everything goes to validation if test_ratio == 0
    if test_ratio == 0:
        return x_train, None, x_temp, y_train, None, y_temp

    val_ratio = 1 - train_ratio - test_ratio

    x_val, x_test, y_val, y_test = train_test_split(
        x_temp,
        y_temp,
        test_size=test_ratio / (test_ratio + val_ratio),
        random_state=SEED
    )

    return x_train, x_test, x_val, y_train, y_test, y_val

In [ ]:
n_features = 4
x, y = make_classification(n_features=n_features, random_state=SEED)

# We don't need a test set since sklearn is probably doing it when fitting the data
x_train, x_test, x_val, y_train, y_test, y_val = split_train_val_test(x, y, train_ratio=0.7, test_ratio=0)

print(x_train[:,1].shape)
print(y_train.shape)

x_all = np.concatenate([x_train[:, i] for i in range(n_features)])
y_all = np.concatenate([y_train for _ in range(n_features)])
# labels = np.concatenate([[f"feature_{i}"] * len(y) for i in range(n_features)])

fig = px.scatter(x=x_all, y=y_all, color=y_all)
fig.show()

(69,)
(69,)


In [ ]:
clf = make_pipeline(StandardScaler(),
                    LinearSVC(random_state=SEED, tol=1e-5))
clf.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('standardscaler', ...), ('linearsvc', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",1e-05


In [ ]:
print(clf.named_steps["linearsvc"].coef_)
print(clf.named_steps["linearsvc"].intercept_)

y_pred = clf.predict(x_val)
acc = accuracy_score(y_val, y_pred)

print(acc)

[[0.14544946 0.4764771  0.59918531 0.44113082]]
[0.15137845]
0.967741935483871
